<a href="https://colab.research.google.com/github/sreejit-19/IQF/blob/main/Abhiroop_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import statsmodels.api as sm

# 1. LOAD THE DATA
df = pd.read_csv('Sorted_Asset_Pricing_Data 2.csv')

# 2. SETUP FACTORS
capm_factors = ['Market_Return']
carhart_factors = ['Market_Return', 'SMB', 'HML', 'WML']
stocks = df['Co_Name'].unique()

alpha_results = []

print(f"Analyzing {len(stocks)} stocks...")

for stock in stocks:
    # Filter for individual stock
    stock_df = df[df['Co_Name'] == stock].dropna()
    y = stock_df['Stock_Return']

    # --- CAPM REGRESSIONS ---
    X_capm = sm.add_constant(stock_df[capm_factors])
    # Standard OLS
    ols_capm = sm.OLS(y, X_capm).fit()
    # HAC (Newey-West) Robust
    hac_capm = sm.OLS(y, X_capm).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

    # --- CARHART REGRESSIONS ---
    X_carhart = sm.add_constant(stock_df[carhart_factors])
    # Standard OLS
    ols_carhart = sm.OLS(y, X_carhart).fit()
    # HAC (Newey-West) Robust
    hac_carhart = sm.OLS(y, X_carhart).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

    # Store Alpha P-values
    alpha_results.append({
        'Stock': stock,
        'CAPM_OLS_p': ols_capm.pvalues['const'],
        'CAPM_HAC_p': hac_capm.pvalues['const'],
        'Carhart_OLS_p': ols_carhart.pvalues['const'],
        'Carhart_HAC_p': hac_carhart.pvalues['const']
    })

# 3. SUMMARIZE RESULTS
df_sig = pd.DataFrame(alpha_results)

# Count Significant Alphas (p < 0.05)
ols_capm_count = (df_sig['CAPM_OLS_p'] < 0.05).sum()
hac_capm_count = (df_sig['CAPM_HAC_p'] < 0.05).sum()
ols_car_count = (df_sig['Carhart_OLS_p'] < 0.05).sum()
hac_car_count = (df_sig['Carhart_HAC_p'] < 0.05).sum()

print("\n--- Significant Alpha Count (p < 0.05) ---")
print(f"CAPM (Naive OLS):  {ols_capm_count}")
print(f"CAPM (HAC Robust): {hac_capm_count}")
print(f"----------------------------------------")
print(f"Carhart (Naive OLS):  {ols_car_count}")
print(f"Carhart (HAC Robust): {hac_car_count}")

# Export for your records
df_sig.to_csv('My_Alpha_Significance_Check.csv', index=False)

Analyzing 50 stocks...

--- Significant Alpha Count (p < 0.05) ---
CAPM (Naive OLS):  0
CAPM (HAC Robust): 0
----------------------------------------
Carhart (Naive OLS):  1
Carhart (HAC Robust): 3


In [ ]:
## shapiro wilk test
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import shapiro, kurtosis, skew

df = pd.read_csv('Sorted_Asset_Pricing_Data 2.csv')
factors = ['Market_Return', 'SMB', 'HML', 'WML']
stocks = df['Co_Name'].unique()

pooled_residuals = []

print("--- Running Shapiro-Wilk Normality Test ---")
for stock in stocks:
    stock_df = df[df['Co_Name'] == stock].dropna()
    y = stock_df['Stock_Return'] - 0.0045  # Excess Return (subtracting RF)
    X = sm.add_constant(stock_df[factors])

    # Fit standard OLS model to get residuals
    model = sm.OLS(y, X).fit()
    pooled_residuals.extend(model.resid.tolist())

    # Individual Stock Test
    stat, p_val = shapiro(model.resid)
    if p_val < 0.05:
        print(f"Stock: {stock:<25} | Shapiro p-value: {p_val:.4f} -> REJECT Normality")

# Pooled Analysis for the entire market
pooled_residuals = np.array(pooled_residuals)
print(f"\n--- Global Residual Market Metrics ---")
print(f"Pooled Skewness: {skew(pooled_residuals):.4f}")
print(f"Pooled Excess Kurtosis: {kurtosis(pooled_residuals):.4f} (Normal = 0)")

--- Running Shapiro-Wilk Normality Test ---
Stock: AARTI INDUSTRIES LTD.     | Shapiro p-value: 0.0298 -> REJECT Normality
Stock: TRIDENT LTD.              | Shapiro p-value: 0.0001 -> REJECT Normality
Stock: ADANI ENTERPRISES LTD.    | Shapiro p-value: 0.0126 -> REJECT Normality
Stock: ALKYL AMINES CHEMICALS LTD. | Shapiro p-value: 0.0063 -> REJECT Normality
Stock: APOLLO HOSPITALS ENTERPRISE LTD. | Shapiro p-value: 0.0157 -> REJECT Normality
Stock: ASAHI INDIA GLASS LTD.    | Shapiro p-value: 0.0220 -> REJECT Normality
Stock: AUROBINDO PHARMA LTD.     | Shapiro p-value: 0.0262 -> REJECT Normality
Stock: LINDE INDIA LTD.          | Shapiro p-value: 0.0000 -> REJECT Normality
Stock: BATA INDIA LTD.           | Shapiro p-value: 0.0375 -> REJECT Normality
Stock: BHARAT HEAVY ELECTRICALS LTD. | Shapiro p-value: 0.0321 -> REJECT Normality
Stock: BHARAT PETROLEUM CORPN. LTD. | Shapiro p-value: 0.0169 -> REJECT Normality
Stock: VODAFONE IDEA LTD.        | Shapiro p-value: 0.0000 -> REJECT No

In [ ]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('Sorted_Asset_Pricing_Data 2.csv')
factors = ['Market_Return', 'SMB', 'HML', 'WML']

# Select an example stock
stock_df = df[df['Co_Name'] == '3M INDIA LTD.'].dropna()
y = stock_df['Stock_Return'] - 0.0045
X = sm.add_constant(stock_df[factors])

# --- THE FIX: Fit OLS model with Newey-West HAC adjustments ---
model_hac = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

# Extract standard errors
hac_standard_errors = model_hac.bse
print("HAC Standard Errors per parameter:")
print(hac_standard_errors)

HAC Standard Errors per parameter:
const            0.010675
Market_Return    0.140089
SMB              0.308053
HML              0.301350
WML              0.156916
dtype: float64


In [ ]:
import pandas as pd
import statsmodels.api as sm

# Define the joint hypothesis string for the 4 factors (excluding intercept)
hypothesis_string = 'Market_Return = 0, SMB = 0, HML = 0, WML = 0'

# Execute the Wald Test using the HAC-fitted model
wald_test_results = model_hac.f_test(hypothesis_string)

print(f"Joint Wald F-Statistic: {wald_test_results.fvalue:.4f}")
print(f"Joint Wald p-value:     {wald_test_results.pvalue:.6e}")
if wald_test_results.pvalue < 0.05:
    print("Decision: Reject H0 -> Factors are JOINTLY significant.")

Joint Wald F-Statistic: 4.7181
Joint Wald p-value:     2.049932e-03
Decision: Reject H0 -> Factors are JOINTLY significant.


In [ ]:
import numpy as np
from numpy.linalg import inv
from scipy.stats import chi2

def hac_wald_test(Gamma_hat, var_hac, k, num_chars, num_factors):

    # Step 1: vectorize Gamma
    theta_hat = Gamma_hat.flatten(order='F')  # vec column-wise

    # Step 2: Build restriction matrix R
    # We want to pick Gamma[k, :] (row k across all factors)

    q = num_factors  # number of restrictions
    R = np.zeros((q, num_chars * num_factors))

    for j in range(num_factors):
        idx = k + j * num_chars   # position in vec(Gamma)
        R[j, idx] = 1

    # Step 3: Compute Wald statistic
    R_theta = R @ theta_hat
    middle = inv(R @ var_hac @ R.T)

    W = R_theta.T @ middle @ R_theta

    # Step 4: p-value
    p_value = 1 - chi2.cdf(W, df=q)

    return W, p_value